In [15]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(base_url="https://www.xiaoyaoapi.com")
model = "claude-sonnet-5"

In [16]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join([block.text for block in message.content if block.type == "text"])

In [27]:
import json


# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)

    # 剥离 Markdown 代码块标记 (```json ... ```)
    cleaned_text = eval_text.strip()
    if cleaned_text.startswith("```json"):
        cleaned_text = cleaned_text[len("```json") :]
    elif cleaned_text.startswith("```"):
        cleaned_text = cleaned_text[3:]
    if cleaned_text.endswith("```"):
        cleaned_text = cleaned_text[:-3]
    cleaned_text = cleaned_text.strip()

    return json.loads(cleaned_text)

In [28]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [29]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [30]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [31]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.666666666666667


In [32]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport re\n\ndef is_valid_s3_bucket_name(name: str) -> bool:\n    \"\"\"\n    Validate an AWS S3 bucket name according to S3 naming rules:\n    - Length between 3 and 63 characters\n    - Only lowercase letters, numbers, hyphens, and periods\n    - Must start and end with a letter or number\n    - No consecutive periods\n    - Must not be formatted as an IP address (e.g., 192.168.5.4)\n    \"\"\"\n    if not isinstance(name, str):\n        return False\n\n    if not (3 <= len(name) <= 63):\n        return False\n\n    # Only lowercase letters, digits, hyphens, and periods allowed\n    if not re.fullmatch(r'[a-z0-9.-]+', name):\n        return False\n\n    # Must start and end with a letter or number\n    if not re.fullmatch(r'[a-z0-9].*[a-z0-9]', name):\n        return False\n\n    # No consecutive periods\n    if '..' in name:\n        return False\n\n    # Must not be formatted as an IP address\n    ip_pattern = r'^\\d{1,3}\\.\\d{1,3}\\.\\d{1,3}\\.\\d{